In [14]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import os

torch.manual_seed(42)


In [15]:
# Save everything next to this file
out_dir = os.getcwd()

In [16]:
DATA_DIR = r"C:\Users\bhoomika m b\digit_recognizer1\data"

transform = transforms.ToTensor()
full_train = datasets.MNIST(DATA_DIR, train=True, download=False, transform=transform)
test_set = datasets.MNIST(DATA_DIR, train=False, download=False, transform=transform)

train_set, val_set = random_split(
    full_train, [50000, 10000], generator=torch.Generator().manual_seed(42)
)
train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
val_loader = DataLoader(val_set, batch_size=256)
test_loader = DataLoader(test_set, batch_size=256)
print("Train:", len(train_set), "| Validation:", len(val_set), "| Test:", len(test_set))

Train: 50000 | Validation: 10000 | Test: 10000


In [17]:
model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 256), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(128, 10),
)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


def evaluate(loader):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            out = model(x)
            total_loss += loss_fn(out, y).item() * len(y)
            correct += (out.argmax(1) == y).sum().item()
            n += len(y)
    return total_loss / n, correct / n


train_losses, val_losses, val_accs = [], [], []
for epoch in range(10):
    model.train()
    running, n = 0.0, 0
    for x, y in train_loader:
        optimizer.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        optimizer.step()
        running += loss.item() * len(y)
        n += len(y)
    train_losses.append(running / n)
    v_loss, v_acc = evaluate(val_loader)
    val_losses.append(v_loss)
    val_accs.append(v_acc)
    print(f"Epoch {epoch + 1:2d}: train loss={train_losses[-1]:.4f}, "
          f"val loss={v_loss:.4f}, val acc={v_acc * 100:.2f}%")

Epoch  1: train loss=0.3561, val loss=0.1716, val acc=94.70%
Epoch  2: train loss=0.1494, val loss=0.1271, val acc=96.21%
Epoch  3: train loss=0.1066, val loss=0.1036, val acc=96.93%
Epoch  4: train loss=0.0847, val loss=0.0918, val acc=97.20%
Epoch  5: train loss=0.0713, val loss=0.0787, val acc=97.61%
Epoch  6: train loss=0.0595, val loss=0.0793, val acc=97.65%
Epoch  7: train loss=0.0521, val loss=0.0827, val acc=97.56%
Epoch  8: train loss=0.0465, val loss=0.0832, val acc=97.44%
Epoch  9: train loss=0.0405, val loss=0.0822, val acc=97.61%
Epoch 10: train loss=0.0399, val loss=0.0861, val acc=97.50%


In [18]:
import os
import numpy as np
import matplotlib.pyplot as plt

out_dir = os.getcwd()

# ---------- Test the model ----------
model.eval()
all_pred, all_true, all_conf, all_img = [], [], [], []
with torch.no_grad():
    for x, y in test_loader:
        probs = torch.softmax(model(x), dim=1)
        c, p = probs.max(1)
        all_pred.append(p)
        all_true.append(y)
        all_conf.append(c)
        all_img.append(x)

pred = torch.cat(all_pred)
true = torch.cat(all_true)
conf = torch.cat(all_conf)
imgs = torch.cat(all_img)

test_acc = (pred == true).float().mean().item()
print(f"Test accuracy: {test_acc * 100:.2f}%")

# ---------- Confusion matrix ----------
cm = np.zeros((10, 10), dtype=int)
for t, p in zip(true.tolist(), pred.tolist()):
    cm[t, p] += 1

plt.figure(figsize=(7, 6))
plt.imshow(cm, cmap="Blues")
plt.colorbar()
plt.xticks(range(10))
plt.yticks(range(10))
plt.xlabel("Predicted digit")
plt.ylabel("True digit")
plt.title("Confusion matrix (test set)")
for i in range(10):
    for j in range(10):
        plt.text(j, i, cm[i, j], ha="center", va="center",
                 color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(out_dir, "confusion_matrix.png"), dpi=120)
plt.show()

# ---------- Precision and recall ----------
print("\nDigit | Precision | Recall")
for d in range(10):
    tp = cm[d, d]
    print(f"  {d}   |   {tp / cm[:, d].sum():.3f}   | {tp / cm[d, :].sum():.3f}")

# ---------- Wrong predictions ----------
wrong = (pred != true).nonzero().flatten()
print(f"\n{len(wrong)} wrong predictions out of {len(true)}")

fig, axes = plt.subplots(2, 5, figsize=(11, 5))
for ax, idx in zip(axes.flat, wrong[:10]):
    ax.imshow(imgs[idx][0], cmap="gray")
    ax.set_title(f"True {true[idx].item()} -> Pred {pred[idx].item()}\n"
                 f"confidence {conf[idx].item() * 100:.0f}%", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(out_dir, "wrong_predictions.png"), dpi=120)
plt.show()

pairs = sorted([(cm[i, j], i, j) for i in range(10) for j in range(10) if i != j],
               reverse=True)
print("\nMost common mistakes:")
for count, t, p in pairs[:5]:
    print(f"  true {t} predicted as {p}: {count} times")

Test accuracy: 97.73%


C:\Users\bhoomika m b\AppData\Local\Temp\ipykernel_96092\1817104222.py:46: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



Digit | Precision | Recall
  0   |   0.989   | 0.980
  1   |   0.992   | 0.981
  2   |   0.991   | 0.974
  3   |   0.967   | 0.984
  4   |   0.980   | 0.978
  5   |   0.961   | 0.979
  6   |   0.977   | 0.981
  7   |   0.965   | 0.987
  8   |   0.971   | 0.966
  9   |   0.978   | 0.963

227 wrong predictions out of 10000

Most common mistakes:
  true 4 predicted as 9: 11 times
  true 9 predicted as 4: 10 times
  true 8 predicted as 5: 10 times
  true 1 predicted as 8: 10 times
  true 9 predicted as 3: 9 times


C:\Users\bhoomika m b\AppData\Local\Temp\ipykernel_96092\1817104222.py:66: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [19]:
DATA_DIR = r"C:\Users\bhoomika m b\digit_recognizer1\data"
full_train = datasets.MNIST(DATA_DIR, train=True, download=False, transform=transform)
test_set = datasets.MNIST(DATA_DIR, train=False, download=False, transform=transform)